## Created by:
- Mikołaj Nowak 151813
- Igor Szymczak 160280

### Task
The association rules are characterized by high support - frequency in the dataset. Can you use this algorithm as a base and try to extract different types of rules:
 - low frequency but strong relation rules e.g. buying Porshe and Rolex is not frequent in the dataset, but usually people who bought Porshe also bought Rolex
 - negative rules e.g. if someone bought low-fat milk it's unlikely there will be whole milk in the basket
 - disjunction e.g. eggs and (kielecki xor winiary ;) )
 - imagine 50% of baskets have milk and 50% of baskets have tea. If there is no relation between them then in ~25% of baskets we will have both. If milk appears together with tea in e.g. 40% of baskets it means there is a pattern. Can you find such rules and use statistical tests to check if the relation is strong?

 Send the report within 144 hours starting from the end of this class to gmiebs@cs.put.poznan.pl; start this email's subject with [IR]


## 1. Preprocessing

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import re

import warnings
warnings.filterwarnings("ignore")

In [ ]:
re.compile(r"'(\w*)'")
data = open("data.txt", 'r').read().split('\n')
data = [re.findall(r"'(\w*)'", s) for s in data]

data = [set(d) for d in data]
objects = list(set().union(*data))
data_binary = [[1 if obj in d else 0 for obj in objects] for d in data]

df = pd.DataFrame(data_binary, columns=objects)

In [260]:
# Dataset is small, so we may generate most of the existing itemsets
# to ensure reasonable sets, we keep only those entries that appear at least 5 times
sup_df = apriori(df, min_support=5/len(df), use_colnames=True).sort_values('support', ascending = False)

## 2. Strong and uncommon rules

In [ ]:
# Contains all association rules with high confidence; we will filter them ;ater
whole_df = association_rules(sup_df)

In [263]:
# We want both products to appear in no more than 2% of transactions
# And that they appear jointly at least 80% of the time
strong_rules = whole_df.loc[(whole_df['antecedent support'] < 0.02) & 
                            (whole_df['consequent support'] < 0.02)]

In [264]:
strong_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
235,"(ketchup, orange, apple)","(grill, banana)",0.006263,0.019833,0.005219,0.833333,42.017544,1.0,0.005095,5.881002,0.982353,0.250000,0.829961,0.548246
236,"(ketchup, apple, banana)","(orange, grill)",0.006263,0.017745,0.005219,0.833333,46.960784,1.0,0.005108,5.893528,0.984874,0.277778,0.830322,0.563725
237,"(ketchup, orange, banana)","(grill, apple)",0.006263,0.013570,0.005219,0.833333,61.410256,1.0,0.005134,5.918580,0.989916,0.357143,0.831041,0.608974
265,"(sausage, ketchup, orange, apple)","(grill, banana)",0.005219,0.019833,0.005219,1.000000,50.421053,1.0,0.005116,inf,0.985310,0.263158,1.000000,0.631579
267,"(sausage, ketchup, apple, banana)","(orange, grill)",0.005741,0.017745,0.005219,0.909091,51.229947,1.0,0.005117,10.804802,0.986142,0.285714,0.907449,0.601604
269,"(ketchup, orange, sausage, banana)","(grill, apple)",0.005741,0.013570,0.005219,0.909091,66.993007,1.0,0.005141,10.850731,0.990761,0.370370,0.907840,0.646853
270,"(ketchup, orange, apple)","(sausage, grill, banana)",0.006263,0.018789,0.005219,0.833333,44.351852,1.0,0.005102,5.887265,0.983613,0.263158,0.830142,0.555556
271,"(ketchup, apple, banana)","(sausage, orange, grill)",0.006263,0.016180,0.005219,0.833333,51.505376,1.0,0.005118,5.902923,0.986765,0.303030,0.830592,0.577957
272,"(ketchup, orange, banana)","(grill, apple, sausage)",0.006263,0.013570,0.005219,0.833333,61.410256,1.0,0.005134,5.918580,0.989916,0.357143,0.831041,0.608974
347,"(butter, chicken)",(wagyu),0.004175,0.006785,0.003653,0.875000,128.961538,1.0,0.003625,7.945720,0.996406,0.500000,0.874146,0.706731


We can see that:
- Many collections seems to be related to doing grill and fruits (they touch very similar things: if you have some of these items: ketchup, mustard, orange, sausage, banana, grill, apple then it is quite probable that others would appear on the right sight). It suggest they people are buying things for barbeque
- All other rules are related to buying wagyu, cheese, butter and chicken (it is quite suprising, as wagyu is really expensive compared to other products on this list and we haven't found any recipe involving these 4 ingredients)

## 3. Negative rules

In [405]:
neg_df = 1 - df 
neg_df = neg_df.rename({c : "not_" + c for c in neg_df.columns}, axis = 1)
neg_df = pd.concat((neg_df, df), axis = 1)

In [406]:
# As there is high chance of not buying most products, we will restrict max element length to 2
neg_sup_df = apriori(neg_df, min_support=5/len(df), max_len=2, use_colnames=True)

In [ ]:
# we will use lift woth threshold = 1.2 to ensure we won't take obvious relations 
# (like x => not y where y is almost never present)
neg_rules_df = association_rules(neg_sup_df, metric='lift', min_threshold=1.2)
neg_rules_df = neg_rules_df[['consequents', 'antecedents', 'support', 'lift']]

In [402]:
neg_rules_df['pos_ante'] = neg_rules_df['antecedents'].apply(lambda x : tuple(obj for obj in tuple(x) if re.match('^[^n]', obj)))
neg_rules_df['pos_cons'] = neg_rules_df['consequents'].apply(lambda x : tuple(obj for obj in tuple(x) if re.match('^[^n]', obj)))
neg_rules_df['neg_cons'] = neg_rules_df['consequents'].apply(lambda x : tuple(obj for obj in tuple(x) if re.match('not *', obj)))
neg_rules_df['neg_ante'] = neg_rules_df['antecedents'].apply(lambda x : tuple(obj for obj in tuple(x) if re.match('not *', obj)))

In [403]:
neg_rules_df['is_valid'] = neg_rules_df.apply(lambda x: ((len(x['pos_ante'])!=0)) & (
                  (len(x['neg_cons'])!=0)) , axis = 1)
neg_rules_df = neg_rules_df.loc[neg_rules_df['is_valid']]
neg_rules_df.drop(columns=["pos_ante" , "pos_cons" , "neg_cons" , "neg_ante" , "is_valid"], inplace=True)

In [404]:
neg_rules_df.iloc[:, [1,0,2, 3]]

,antecedents,consequents,support,lift
26,(orange),"(apple, not_ketchup)",0.056889,2.468839
27,(apple),"(orange, not_ketchup)",0.056889,2.457798
30,(banana),"(apple, not_ketchup)",0.056367,2.399889
31,(apple),"(not_ketchup, banana)",0.056367,2.443618
34,(sausage),"(not_ketchup, mustard)",0.058977,3.012243
...,...,...,...,...
635,(mustard),"(grill, not_chicken)",0.031315,5.886329
638,(orange),"(not_chicken, banana)",0.038100,2.392215
639,(banana),"(orange, not_chicken)",0.038100,2.372172
642,(grill),"(sausage, not_chicken)",0.061065,3.689467


We can see 2 important negative rule that is:
- buying 1 meat decreases chance of buying another meat
- buying butter decreases chances of buying milk (yet it has low support, so we should be careful)

## 4. Exclusive occurence (XOR)

We will consider following detection pattern:
A xor B if and only if:
- sup(A) > threshold
- sup(B) > threshold
- sup(A, B) < (1/2) * sup(A) * sup(B)

In [265]:
from itertools import combinations

In [266]:
# For faster data recovery, we will transform sup_df into dict
xor_dict = {}
for row in sup_df.itertuples():
    xor_dict[frozenset(row[2])] = row[1]

In [267]:
xor_rules = pd.DataFrame(columns=["item_1", "item_2", "support_1", "support_2", 
                                  "support_conjunction", "expected_support"])
for i,j in combinations(xor_dict.keys(), 2):
    joined_set = frozenset(frozenset.union(i, j))
    joint_support = xor_dict.get(joined_set)
    if(joint_support == None or xor_dict[i]*xor_dict[j] < 2*joint_support):
        continue
    xor_rules.loc[-1] = [i, j, xor_dict[i], xor_dict[j], 
                         joint_support, xor_dict[i]*xor_dict[j]]
    xor_rules.index = xor_rules.index + 1
xor_rules = xor_rules.sort_index()

In [268]:
xor_rules

,item_1,item_2,support_1,support_2,support_conjunction,expected_support
0,"(milk, pork)","(chocolate, chicken)",0.088205,0.074113,0.002610,0.006537
1,"(yogurt, chicken)","(milk, pork)",0.093424,0.088205,0.002610,0.008240
2,(eggs),"(sausage, mustard)",0.095511,0.060543,0.002610,0.005783
3,"(sausage, chicken)","(cheese, bread)",0.102296,0.052192,0.002610,0.005339
4,"(milk, chicken)","(pork, chocolate)",0.149791,0.037578,0.002610,0.005629
5,"(milk, chicken)","(milk, beef)",0.149791,0.047495,0.002610,0.007114
6,"(milk, chicken)","(yogurt, pork)",0.149791,0.051148,0.002610,0.007662
7,"(milk, chicken)","(milk, pork)",0.149791,0.088205,0.005219,0.013212
8,"(milk, chicken)",(beef),0.149791,0.127871,0.002610,0.019154
9,(banana),"(beef, pork)",0.165449,0.034447,0.002610,0.005699


We can see that:
-  Meets are usually exclusive (Seems reasonable); Most of this rules probably come down to this
-  Eggs are exclusive with (sausage, mustard); we don't know why that could be
- banana is exclusive with (beef, pork); maybe because buying a lot of meat means we don't buy fruits; yet we lack other fruits (oranges, apples).
- bread is exclusive with (sausage, eggs); a bit strange, we assumed usually you have some bread to sausage and eggs

## 5. Cooccurence

coocurence is characterized by lift metric; if lift>1, then association is positive. \
For readability concerns, we will only take items with lift > 2 qnd where both antecedent and consequent has 1 element \
We will use fisher_exact test from scipy library to verify that association is significant


In [274]:
lift_df = association_rules(sup_df, metric='lift', min_threshold=2.0)
amount = len(data)

In [270]:
lift_df = lift_df[["antecedents" , "consequents" , "antecedent support" , "consequent support" , "lift"]]
lift_df['antecedent support'] *=amount
lift_df['consequent support'] *=amount

In [282]:
from scipy.stats import chi2_contingency
def rule_significance(df, antecedent, consequent):
    # We will care only about cooccurence of 1-element sets
    # if you wish to change it, remove if below (yet it would take much longer)
    if(len(antecedent) > 1 or len(consequent) > 1):
        return 1

    A_items = list(antecedent)
    B_items = list(consequent)
    
    
    A_mask = df[A_items].all(axis=1)
    B_mask = df[B_items].all(axis=1)
    
    contingency = pd.crosstab(A_mask, B_mask)
    
    # choose method
    _, p, _, _ = chi2_contingency(contingency)
    
    return p

p_values = []
for row in lift_df.itertuples():
    p = rule_significance(df, row[1], row[2])
    p_values.append(p)

lift_df['p_value'] = p_values
lift_df['significant'] = lift_df['p_value'] < 0.01
lift_df = lift_df.loc[lift_df['significant'] == True].sort_values(by='lift', ascending=False)
# filter duplicates
lift_df = lift_df[["antecedents" , "consequents", "lift"]]
lift_df["connected"] = lift_df[["antecedents" , "consequents"]].apply(lambda x : x.iloc[0].union(x.iloc[1]), axis = 1)
lift_df.drop_duplicates(subset=['connected'], inplace=True)
lift_df.drop(columns='connected', inplace=True)
lift_df.rename({'antecedents' : 'item_1', 'consequents' : 'item_2'}, axis = 1, inplace=True)

In [283]:
lift_df

,item_1,item_2,lift
2030,(butter),(wagyu),105.274725
27,(grill),(ketchup),6.932247
16,(grill),(mustard),6.266092
0,(grill),(sausage),3.746970
18,(ketchup),(sausage),3.326620
9,(mustard),(sausage),3.012415
3,(apple),(orange),2.561695
6,(apple),(banana),2.492439
4,(orange),(banana),2.332153


We can see that:
- wagyu suprisingly often cooccurs with butter (maybe people use it to prepare wagyu? also it is possible it is wagyu very uncommon so a simple coincidence could have caused that)
- grill coocurs with ketchup, mustard, sausage (probably people who buy grill want to grill sausages)
- sausage coocurs with ketchup/mustard (obvious why)
- Fruits cooccur together